[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/6_optimizacion/_assets/01_impresora_lineal.ipynb)

# Notebook 1 · La impresora

Acompaña a la **clase 1** de la unidad de modelado y optimización. Da por leídas
las páginas 1 a 5: *Leer la bitácora*, *Escribir el modelo*, *El dibujo*,
*Qué es una respuesta* y *Patrones lineales*.

Aquí no hay teoría nueva. Hay tres cosas:

1. El modelo de la impresora escrito como lo entiende un programa, y resuelto.
2. El mismo dibujo de la página 3, hecho por la computadora.
3. **Una bitácora nueva, al final, para que la modeles tú.** Eso es lo que
   importa.

Corre las celdas en orden con `Shift + Enter`. No hay nada que entregar.


In [ ]:
# La convención de signos, que es la fuente número uno de errores al pasar
# del papel al código. Es la misma tabla de la página 5.
#
#   - En las notas, la constante va a la derecha y se admiten <=, >= y =.
#   - Al solver se le entrega TODO con <=:  a·x >= b  se escribe  -a·x <= -b.
#   - linprog MINIMIZA. Un máximo se resuelve con -c, y al valor que devuelve
#     hay que cambiarle el signo OTRA VEZ.
#   - Las cotas simples sobre una variable van en `bounds`, no como una fila
#     más de A_ub. En linprog la cota por omisión ya es (0, inf).

import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.optimize import linprog

print('listo')


In [ ]:
# --- El modelo de la impresora, tal como salió del lienzo de la página 2 ---
#
#   max  4 x1 + 3 x2                      créditos que abona el depósito
#   s.a.   x1 +   x2 <= 10                horas de impresora
#        2 x1 +   x2 <= 18                polímero, en kg
#          x1 + 2 x2 <= 18                energía, en kWh  (supuesto)
#          x1, x2 >= 0

c = np.array([4, 3])                       # créditos por filtro y por celda
A = np.array([[1, 1], [2, 1], [1, 2]])     # horas, polímero, energía
b = np.array([10, 18, 18])
recursos = ['horas', 'polímero', 'energía']

r = linprog(c=-c, A_ub=A, b_ub=b, method='highs')   # ojo: -c, porque MINIMIZA

print('plan óptimo :', r.x)
print('fun devuelto:', r.fun, '  <- viene negado')
print('créditos    :', -r.fun)

# Los assert van con isclose, NUNCA con ==: el solver trabaja con flotantes y
# devuelve 47.99999999999999 donde la página dice 48.
assert np.allclose(r.x, [8, 2]), r.x
assert np.isclose(-r.fun, 38), -r.fun
print('\ncuadra con la página 3: (8, 2) con 38 créditos')


In [ ]:
# --- El dibujo de la página 3, hecho por la computadora ---
#
# Las esquinas se encuentran con el mismo procedimiento que enseña la página:
# cruzar las rectas de dos en dos y quedarse solo con los cruces que cumplen
# TODAS las desigualdades. Ese segundo paso no es opcional.

def esquinas(A, b):
    filas = np.vstack([A, [-1, 0], [0, -1]])
    lados = np.concatenate([b, [0, 0]])
    puntos = []
    for i, j in combinations(range(len(filas)), 2):
        M = filas[[i, j]]
        if abs(np.linalg.det(M)) < 1e-9:      # rectas paralelas: no se cruzan
            continue
        p = np.linalg.solve(M, lados[[i, j]])
        if np.all(filas @ p <= lados + 1e-9):  # ¿cae dentro de todo lo demás?
            puntos.append(p)
    P = np.unique(np.round(puntos, 9), axis=0) + 0.0
    P[np.abs(P) < 1e-9] = 0.0                  # quita los -0. del solve
    centro = P.mean(axis=0)                    # orden antihorario, para pintar
    return P[np.argsort(np.arctan2(*(P - centro).T[::-1]))]

V = esquinas(A, b)
print('esquinas:'); print(V)
print('créditos:', V @ c)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.fill(V[:, 0], V[:, 1], alpha=.25, label='planes que se pueden hacer')
t = np.linspace(0, 11, 200)
for k in range(3):
    if A[k, 1]:
        ax.plot(t, (b[k] - A[k, 0] * t) / A[k, 1], lw=1.6, label=recursos[k])

z_opt = -r.fun
for v in (12, 24, z_opt, 48):                  # la familia de curvas de nivel
    gana = np.isclose(v, z_opt)
    ax.plot(t, (v - c[0] * t) / c[1], ls='-' if gana else '--',
            lw=2.6 if gana else 1.1, color='crimson' if gana else 'gray',
            label=f'4x1 + 3x2 = {v:.0f}' if gana else None)

ax.plot(*r.x, 'o', ms=11, color='crimson')
ax.annotate(f'({r.x[0]:.0f}, {r.x[1]:.0f})', r.x, textcoords='offset points',
            xytext=(12, 8), fontsize=12, color='crimson')
ax.set(xlim=(0, 11), ylim=(0, 11), xlabel='x1  filtros', ylabel='x2  celdas',
       title='la última curva de nivel que todavía toca')
ax.grid(alpha=.25); ax.legend(loc='upper right', fontsize=9)
plt.show()


In [ ]:
# --- ¿Cuánto más ganarías con una unidad más de cada recurso? ---
#
# Se calcula RESOLVIENDO OTRA VEZ con una hora, un kilo o un kilowatt-hora de
# más, que es la definición. No se le pide al solver: `r.ineqlin.marginals`
# devuelve estos mismos números NEGADOS, porque son los del problema que el
# solver minimiza, y ahí ya se ha tropezado más de uno.

z0 = -r.fun
print(f'{"recurso":10} {"gastado":>8} {"hay":>5} {"sobra":>6} {"vale una más":>14}\n')
for k in range(3):
    bb = b.copy(); bb[k] += 1
    rk = linprog(c=-c, A_ub=A, b_ub=bb, method='highs')
    gastado = A[k] @ r.x
    print(f'{recursos[k]:10} {gastado:8.0f} {b[k]:5d} {b[k]-gastado:6.0f}'
          f' {-rk.fun - z0:14.2f}')

precios = [round(-linprog(c=-c, A_ub=A, b_ub=b + np.eye(3, dtype=int)[k],
                          method='highs').fun - z0, 6) for k in range(3)]
assert np.allclose(precios, [2, 1, 0]), precios
print('\nla energía vale 0 porque sobran 6 kWh: de nada sirve tener más')
print('lo que el solver devuelve, para que lo veas:', r.ineqlin.marginals)


---

## Ahora tú: el invernadero

Otro rincón de la nave, el mismo trabajo. **Esta historia no está resuelta en
ninguna página**: es el ejercicio.

### La situación

En la cubierta baja hay un **invernadero hidropónico**. Quedaron **cuatro
charolas libres** y hay que sembrarlas hoy, antes de llegar al mismo depósito.

Se pueden sembrar dos cosas, y solo dos:

- **Tubérculo**, que el depósito paga a **8 créditos** la charola.
- **Hoja**, que paga a **cinco**.

El tubérculo se paga mejor, así que la tentación es sembrar puro tubérculo. Otra
vez no se puede, y otra vez por lo mismo: **tres cosas se acaban**. Las horas de
**luz** de crecimiento que quedan antes del ciclo de sueño, el **sustrato** que
hay en el almacén, y el **agua** que la nave destina al invernadero.

Y los dos cultivos **no gastan lo mismo**. Uno es lento y sobrio con el agua; el
otro crece rápido y bebe el doble.

### Lo que llegó

Igual que con la impresora, nadie te da eso ordenado. Llega así:

> **Bitácora del invernadero.** Quedaron 4 charolas libres y hay que
> decidir qué sembrar antes de la parada. En el depósito pagan 8 créditos por
> cada charola de tubérculo y cinco por cada una de hoja.
>
> De luz de crecimiento nos quedan 10 horas antes del ciclo de sueño. El
> tubérculo se lleva 2 horas y la hoja 1.
>
> De sustrato hay 7 kilos, y cualquiera de los dos cultivos se lleva uno.
>
> De agua tenemos la de siempre. El tubérculo bebe 1 litro.
>
> El invernadero está a 22 grados, como toda la semana.
>
> Se me olvidaba la hoja: de agua bebe 2 litros, el doble que el tubérculo.
>
> Y el biólogo insiste en que no conviene sembrar puro tubérculo.

| | En esta historia |
|---|---|
| **Quién decide** | La tripulación, hoy, antes de llegar al depósito |
| **Qué decide** | Cuántas charolas de tubérculo y cuántas de hoja sembrar |
| **Qué lo limita** | Tres cosas que se acaban: luz, sustrato y agua |
| **Qué se quiere** | Que el total de créditos sea lo más grande posible |
| **Qué estorba** | La bitácora trae frases que no son ninguna de las cuatro anteriores |

### Las cinco cosas que hay que hacer

1. **Escribe la tabla de recursos.** Marca **qué número sobra**, **cuál falta** y
   **qué frase es ambigua**. Son las tres trampas de la página 1, y esta bitácora
   tiene una de cada.
2. **Escribe el modelo**, recorriendo los siete pasos del lienzo: variables con
   su dominio, objetivo, restricciones.
3. **Resuélvelo** en la celda de abajo.
4. **Dibújalo**, reusando la función `esquinas` de más arriba.
5. **Interroga tu respuesta.** ¿Qué recurso **sobra** en el plan óptimo? Y tu
   supuesto del paso 1, **¿hasta dónde aguanta?**

*Pista para la 5: tacha del modelo el recurso que supusiste y resuelve otra vez.
Si el resultado no cambia, ése es el techo, y el umbral es lo que el óptimo
consume de ese recurso. Es el argumento del 12 de la página 3, con otros
números.*


In [ ]:
# --- Tu modelo del invernadero ---
#
# Está sembrado con los números DLA IMPRESORA, no con los tuyos. Así corre de
# entrada y te devuelve 38, que es visiblemente la respuesta de otro problema.
# Cámbialos por los del invernadero.

c_inv = np.array([4, 3])                          # <- créditos por charola
A_inv = np.array([[1, 1], [2, 1], [1, 2]])        # <- luz, sustrato, agua
b_inv = np.array([10, 18, 18])                    # <- de cuánto dispones
recursos_inv = ['horas', 'polímero', 'energía']   # <- cómo se llaman

r_inv = linprog(c=-c_inv, A_ub=A_inv, b_ub=b_inv, method='highs')
print('plan     :', r_inv.x)
print('créditos :', -r_inv.fun)
print('esquinas :'); print(esquinas(A_inv, b_inv))

for k, nombre in enumerate(recursos_inv):
    print(f'{nombre:10} gastas {A_inv[k] @ r_inv.x:5.1f} de {b_inv[k]:3d}'
          f'   sobra {b_inv[k] - A_inv[k] @ r_inv.x:5.1f}')
